## Step 0: Read Files

In [1]:
# read data
import pandas as pd

df_ptt = pd.read_csv("bda2025_202301-202503_討論數據_ptt.csv", encoding="utf-8")
df_dcard1 = pd.read_csv("bda2025_202301-202503_討論數據_dcard_1.csv", encoding="utf-8")
df_dcard2 = pd.read_csv("bda2025_202301-202503_討論數據_dcard_2.csv", encoding="utf-8")
df_mobile = pd.read_csv("bda2025_202301-202503_討論數據_mobile01.csv", encoding="utf-8")
df_news = pd.read_csv("bda2025_202301-202503_內容數據_新聞.csv", encoding="utf-8")
df_listed_dict = pd.read_excel("bda2025_stock_202301-202503_上市.xlsx", sheet_name=None)
df_OTC = pd.read_excel("bda2025_stock_202301-202503_上櫃.xlsx")

FileNotFoundError: [Errno 2] No such file or directory: 'bda2025_202301-202503_討論數據_ptt.csv'

In [ ]:
listed_list = []

for df in df_listed_dict.values():
    listed_list.append(df)

df_listed = pd.concat(listed_list, ignore_index=True)
df_listed

In [ ]:
print(df_ptt.columns)
print(df_dcard1.columns)
print(df_dcard2.columns)
print(df_mobile.columns)
print(df_news.columns)

## Step 1: 先選出要預測的股票標的
- 篩選條件：股價波動大的股票
- 篩選方式：不知道他報酬率怎麼算出來的，但就是找出 Variance 報酬率最大的股票

### 1.1 找出股票波動最大的前50檔股票

In [ ]:
variance_by_stock = df_listed.groupby("名稱")["報酬率％"].var().sort_values(ascending=False)

top50 = variance_by_stock.head(50)
for stock, variance in top50.items():
    stock = stock.strip()
    print(f"股票: {stock:7}", end="")
    print(f"報酬率變異數: {variance:.2f}")

stock_lists = []

for stock in top50.index:
    stock = stock.strip()
    stock_lists.append(stock)

### 1.2 再算出這五十檔股票的 term-frequency

In [ ]:
import pandas as pd

class Source:
    def __init__(self, name, df):
        self.name = name
        self.df = df

class Company:
    def __init__(self, name):
        self.name = name

    def count_mentions_in(self, source):
        count = 0
        for content in source.df["content"]:
            if isinstance(content, str) and (self.name in content):
                count += 1
        return count

# Create source and company objects
sources = [
    Source("PTT", df_ptt),
    Source("Dcard1", df_dcard1),
    Source("Dcard2", df_dcard2),
    Source("Mobile01", df_mobile),
    Source("News", df_news)
]

companies = [Company(name) for name in stock_lists]

# Build a list of dicts (rows)
test_data = []
for company in companies:
    row = {"公司": company.name}
    total = 0
    for source in sources:
        count = company.count_mentions_in(source)
        row[source.name] = count
        total += count
    row["總計"] = total
    test_data.append(row)

summary_df = pd.DataFrame(test_data)
summary_df.sort_values(by="總計", ascending=False, inplace=True)

# drop=Ture 會讓原本的 index 消掉，不然就會有兩欄 index
summary_df.reset_index(drop=False, inplace=True, names="old")
summary_df

## Step 2：Data Processing

### Step 2.1 從五個文章來源，篩選出所有 title, content 包含 company name 的文章 id 並存成一個 list

In [ ]:
def get_fileID_contain_companyname(companyname) -> list:
    # 先篩出包含公司名稱的資料 ID
    fileID_contain_companyname = []

    for source in sources:
        for id, title, content in zip(source.df['id'], source.df["title"], source.df["content"]):
            if isinstance(title, str) and isinstance(content, str):
                title = title.strip()
                content = content.strip()
                if (companyname in title) or (companyname in content):
                    fileID_contain_companyname.append(id)
    
    if len(fileID_contain_companyname) == 0:
        print(f"沒有包含 {companyname} 的資料 ID")
        raise Exception(f"沒有包含 {companyname} 的資料 ID")
                
    return fileID_contain_companyname

In [ ]:
fileID_contain_companyname = get_fileID_contain_companyname("龍德造船")
print(fileID_contain_companyname)

### Step 2.2 清理 dataframe

In [ ]:
def create_cleaned_df(fileID_contain_companyname, companyname) -> pd.DataFrame:
    import os
    if os.path.exists(f"cleaned_df_{companyname}.csv"):
        return pd.read_csv(f"cleaned_data/cleaned_data_{companyname}.csv", encoding="utf-8-sig")


    # 按照 ID 製作新的 dataframe
    cleaned_ppt_df = df_ptt[df_ptt["id"].isin(fileID_contain_companyname)].copy()
    cleaned_dcard1_df = df_dcard1[df_dcard1["id"].isin(fileID_contain_companyname)].drop(columns=["content_type"]).copy()
    cleaned_dcard2_df = df_dcard2[df_dcard2["id"].isin(fileID_contain_companyname)].drop(columns=["content_type"]).copy()
    cleaned_mobile_df = df_mobile[df_mobile["id"].isin(fileID_contain_companyname)].drop(columns=["content_type"]).copy()
    cleaned_news_df = df_news[df_news["id"].isin(fileID_contain_companyname)].copy()
    cleaned_ppt_df['source'] = 'PTT'
    cleaned_dcard1_df['source'] = 'Dcard1'
    cleaned_dcard2_df['source'] = 'Dcard2'
    cleaned_mobile_df['source'] = 'Mobile01'
    cleaned_news_df['source'] = 'News'
    cleaned_df = pd.concat([cleaned_ppt_df, cleaned_dcard1_df, cleaned_dcard2_df, cleaned_mobile_df, cleaned_news_df], ignore_index=True)

    import re
    # 清除 title, content 中的表點符號、英文、數字；將日期轉換格式
    def clean_text(text):
        """Remove punctuation, English letters, and digits"""
        text = re.sub(r"[^\u4e00-\u9fa5]", "", text)
        return text
    
    cleaned_df["title"] = cleaned_df["title"].apply(clean_text)
    cleaned_df["content"] = cleaned_df["content"].apply(clean_text)

    def clean_time(time):
        time = time.split(" ")
        return time[0]

    # 將時間去掉小時、分鐘、秒只留下日期
    try:
        cleaned_df["post_time"] = cleaned_df["post_time"].apply(clean_time)
    except:
        pass

    # 將 "post_time" 也轉換為 datetime 格式
    cleaned_df["post_time"] = pd.to_datetime(cleaned_df["post_time"], format="%Y-%m-%d")
    # 新增一欄 "mark"，預設為 0
    cleaned_df['mark'] = 0

    cleaned_df.to_csv(f"cleaned_data/cleaned_data_{companyname}.csv", index=False, encoding="utf-8-sig")
    
    return cleaned_df

In [ ]:
cleaned_df = create_cleaned_df(fileID_contain_companyname, companyname="龍德造船")
cleaned_df

### Step 2.3.1 將文章標記成看漲文章或是看跌文章

In [ ]:
def create_df_listed_selected(df_listed, companyname):
    # 先將 "年月日" 轉換為 datetime 格式
    df_listed_selected = df_listed[df_listed["名稱"] == companyname]
    df_listed_selected = df_listed_selected[["年月日", "報酬率％"]]
    df_listed_selected["年月日"] = pd.to_datetime(df_listed_selected["年月日"], format="%Y/%m/%d")
    return df_listed_selected

In [ ]:
def update_mark(cleaned_df, df_listed_selected):
    for idx, (date, mark) in enumerate(zip(cleaned_df['post_time'], cleaned_df['mark'])):
        row = df_listed_selected[df_listed_selected['年月日'] == date]
        
        if row.empty:
            continue
        
        new_mark = int(row['報酬率％'].values[0] > 0)
        
        cleaned_df.loc[idx, 'mark'] = new_mark

    return cleaned_df

### Step 2.4.1 用 term frequency 抓出 keyword

In [ ]:
def get_keywords(cleaned_df, keywordnum, companyname) -> list:
    import os
    import pandas as pd
    from collections import Counter
    from tqdm import tqdm

    # 檢查是否已經存在關鍵字檔案
    if os.path.exists(f"keywords/{keywordnum}Keywordsof_{companyname}.csv"):
        return pd.read_csv(f"keywords/{keywordnum}Keywordsof_{companyname}.csv", encoding="utf-8-sig")["word"].tolist()

    def char_tokenize(text):
        return list(text)

    def generate_ngrams(tokens, n):
        for i in range(len(tokens) - n + 1):
            yield ''.join(tokens[i : i + n])

    # Initialize Counter
    ngram_counter = Counter()

    # tqdm progress bar on content rows
    for content in tqdm(cleaned_df["content"].dropna().astype(str), desc="Processing N-grams"):
        tokens = char_tokenize(content)
        for n in [2, 3, 4]:
            ngram_counter.update(generate_ngrams(tokens, n))

    # Create DataFrame from Counter
    frequencies_df = pd.DataFrame(ngram_counter.items(), columns=["word", "term_frequency"])
    frequencies_df.sort_values(by="term_frequency", ascending=False, inplace=True)
    frequencies_df.reset_index(drop=True, inplace=True)

    # Save and return top keywords
    output_path = f"keywords/{keywordnum}Keywordsof_{companyname}.csv"
    frequencies_df.head(keywordnum).to_csv(output_path, index=False, encoding="utf-8-sig")

    return frequencies_df.head(keywordnum)["word"].tolist()

### Step 2.4.2 用 TFIDF 抓出 keyword

In [ ]:
#TF-IDF
def get_keywords_tfidf(cleaned_df, keywordnum, companyname) -> list:
    import os
    import pandas as pd
    from sklearn.feature_extraction.text import TfidfVectorizer

    # 檢查是否已經存在 TF-IDF 檔案
    if os.path.exists(f"keywords/{keywordnum}TFIDFKeywordsof_{companyname}.csv"):
        return pd.read_csv(f"keywords/{keywordnum}TFIDFKeywordsof_{companyname}.csv", encoding="utf-8-sig")["word"].tolist()

    # Prepare the corpus
    corpus = cleaned_df["content"].dropna().astype(str).tolist()

    # Initialize TfidfVectorizer with char-level 2~4 grams
    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
    X = vectorizer.fit_transform(corpus)  # Returns a sparse matrix

    # Get feature names and their max TF-IDF score across all documents
    feature_names = vectorizer.get_feature_names_out()
    max_tfidf_scores = X.max(axis=0).toarray().flatten()

    tfidf_df = pd.DataFrame({
        "word": feature_names,
        "tfidf": max_tfidf_scores
    })

    tfidf_df.sort_values(by="tfidf", ascending=False, inplace=True)
    tfidf_df.reset_index(drop=True, inplace=True)

    # Save and return top N keywords
    tfidf_df.head(keywordnum).to_csv(f"keywords/{keywordnum}TFIDFKeywordsof_{companyname}.csv", index=False, encoding="utf-8-sig")
    return tfidf_df.head(keywordnum)["word"].tolist()

### Step 2.4.3 用 Lift 抓出 keyword

In [ ]:
def get_keywords_lift(cleaned_df, keywordnum, companyname) -> list:
    import os
    import pandas as pd
    from collections import Counter
    from sklearn.feature_extraction.text import CountVectorizer

    # 檢查是否已經存在 Lift 檔案
    if os.path.exists(f"keywords/{keywordnum}LiftKeywordsof_{companyname}.csv"):
        return pd.read_csv(f"keywords/{keywordnum}LiftKeywordsof_{companyname}.csv", encoding="utf-8-sig")["word"].tolist()

    # Prepare corpus and labels
    cleaned_df = cleaned_df.dropna(subset=["content"])
    corpus_all = cleaned_df["content"].astype(str).tolist()
    corpus_pos = cleaned_df[cleaned_df["mark"] == 1]["content"].astype(str).tolist()

    # Use CountVectorizer with char n-grams
    vectorizer = CountVectorizer(analyzer="char", ngram_range=(2, 4))
    X_all = vectorizer.fit_transform(corpus_all)
    X_pos = vectorizer.transform(corpus_pos)

    # Compute term frequencies
    word_counts_all = X_all.sum(axis=0).A1
    word_counts_pos = X_pos.sum(axis=0).A1
    vocab = vectorizer.get_feature_names_out()

    total_all = word_counts_all.sum()
    total_pos = word_counts_pos.sum()

    # Compute lift = P(w | pos) / P(w)
    lift_scores = (word_counts_pos / total_pos) / (word_counts_all / total_all)

    lift_df = pd.DataFrame({
        "word": vocab,
        "lift": lift_scores
    })

    lift_df.sort_values(by="lift", ascending=False, inplace=True)
    lift_df.reset_index(drop=True, inplace=True)

    # Save and return top N keywords
    lift_df.head(keywordnum).to_csv(f"keywords/{keywordnum}LiftKeywordsof_{companyname}.csv", index=False, encoding="utf-8-sig")
    return lift_df.head(keywordnum)["word"].tolist()

### Step 2.5 建構向量空間 final data
針對每篇文章，計算不同關鍵字出現的次數


In [ ]:
from typing import Literal
def create_final_data(cleaned_df, key_words, companyname, keyword_method, method: Literal["count", "contain"] = "count") -> pd.DataFrame:
    import pandas as pd
    from tqdm import tqdm

    # Step 1: Copy essential columns
    df = cleaned_df[["id", "post_time", "mark"]].copy()

    # Step 2: Initialize all keyword columns at once with 0s
    zeros_df = pd.DataFrame(0, index=df.index, columns=key_words)
    df = pd.concat([df, zeros_df], axis=1)

    # Step 3: Only operate on rows where content is a string
    valid_content = cleaned_df["content"].fillna("")

    # Step 4: Vectorized keyword counting
    if method == "count":
        for key in tqdm(key_words, desc="Counting keywords"):
            df[key] = valid_content.str.count(key)
    elif method == "contain":
        for key in tqdm(key_words, desc="Checking keyword containment"):
            df[key] = valid_content.str.contains(key).astype(int)
    
    df.to_csv(f"finaldata/final_data_{method}_{keyword_method}_{len(key_words)}_{companyname}.csv", index=False, encoding="utf-8-sig")

    return df

## Step 3: 模型建置

In [ ]:
# def run_models(df, run_PCA=True, PCA_n_components=100, test_size=0.2) -> dict:
#     from sklearn.model_selection import train_test_split
#     from sklearn.preprocessing import StandardScaler
#     from sklearn.naive_bayes import GaussianNB
#     from sklearn.neighbors import KNeighborsClassifier
#     from sklearn.svm import SVC
#     from sklearn.tree import DecisionTreeClassifier
#     from sklearn.ensemble import RandomForestClassifier
#     from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
#     import matplotlib.pyplot as plt
#     import seaborn as sns

#     # Example: Assume the target column is named 'target'
#     X = df.drop(['mark', 'id', 'post_time'], axis=1)  # Features
#     y = df['mark']              # Target

#     # 1. Split into training and testing sets
#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

#     # Optional: Standardize features
#     scaler = StandardScaler()
#     X_train = scaler.fit_transform(X_train)
#     X_test = scaler.transform(X_test)


#     if run_PCA:
#         # Apply PCA
#         from sklearn.decomposition import PCA
#         pca = PCA(n_components=PCA_n_components)  # You can also try n_components=20 for a fixed value
#         X_train = pca.fit_transform(X_train)
#         X_test = pca.transform(X_test)

#         print(f"PCA reduced the feature count from {X.shape[1]} to {X_train.shape[1]}")

#     # 2. Initialize classifiers
#     models = {
#         'Naive Bayes': GaussianNB(),
#         'KNN': KNeighborsClassifier(n_neighbors=5),
#         'SVM': SVC(),
#         'Decision Tree': DecisionTreeClassifier(random_state=42),
#         'Random Forest': RandomForestClassifier(random_state=42)
#     }

#     # Store accuracy results
#     accuracies = {}

#     # 3. Train, evaluate, and plot confusion matrix
#     for name, model in models.items():
#         if name in ['KNN', 'SVM']:
#             model.fit(X_train, y_train)
#             y_pred = model.predict(X_test)
#         else:
#             model.fit(X_train, y_train)
#             y_pred = model.predict(X_test)

#         acc = accuracy_score(y_test, y_pred)
#         accuracies[name] = acc

        # print(f"\n{name} Classifier:")
        # print("Accuracy:", acc)
        # print(classification_report(y_test, y_pred))

        # # Confusion Matrix
        # cm = confusion_matrix(y_test, y_pred)
        # plt.figure(figsize=(6, 4))
        # sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        # plt.title(f'{name} - Confusion Matrix')
        # plt.xlabel('Predicted')
        # plt.ylabel('Actual')
        # plt.tight_layout()
        # plt.show()

    # # 4. Accuracy Comparison Plot with labels
    # plt.figure(figsize=(8, 5))
    # ax = sns.barplot(x=list(accuracies.keys()), y=list(accuracies.values()))

    # # Add labels on top of bars
    # for i, bar in enumerate(ax.patches):
    #     height = bar.get_height()
    #     ax.text(
    #         bar.get_x() + bar.get_width() / 2,
    #         height + 0.02,  # slight offset above bar
    #         f'{height:.2f}',
    #         ha='center',
    #         va='bottom',
    #         fontsize=10
    #     )

    # plt.ylabel('Accuracy')
    # plt.title('Model Accuracy Comparison')
    # plt.ylim(0, 1.05)
    # plt.xticks(rotation=45)
    # plt.tight_layout()
    # plt.show()

    # from scipy.stats import mode

    # # 5. Poll-Based Ensemble from KNN, SVM, Decision Tree, and Random Forest
    # # Ensure you have access to their predictions
    # poll_models = ['KNN', 'SVM', 'Decision Tree', 'Random Forest']
    # poll_preds = []

    # for name in poll_models:
    #     model = models[name]
    #     if name in ['KNN', 'SVM']:
    #         pred = model.predict(X_test)
    #     else:
    #         pred = model.predict(X_test)
    #     poll_preds.append(pred)
    
    # print("Poll predictions:", poll_preds)

    # # Stack predictions and take the mode (majority vote) along axis=0
    # import numpy as np
    # ensemble_preds = mode(np.array(poll_preds), axis=0, keepdims=False).mode[0]

    # print("Ensemble predictions:", ensemble_preds)

    # # Evaluate ensemble result
    # ensemble_acc = accuracy_score(y_test, y_pred=ensemble_preds)
    # accuracies['Poll-Based Ensemble'] = ensemble_acc

    # print("\nPoll-Based Ensemble Classifier:")
    # print("Accuracy:", ensemble_acc)
    # print(classification_report(y_test, ensemble_preds))

    # # Confusion Matrix for Ensemble
    # cm = confusion_matrix(y_test, ensemble_preds)
    # plt.figure(figsize=(6, 4))
    # sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    # plt.title('Poll-Based Ensemble - Confusion Matrix')
    # plt.xlabel('Predicted')
    # plt.ylabel('Actual')
    # plt.tight_layout()
    # plt.show()

    # # 6. Re-plot Accuracy Comparison with Ensemble included
    # plt.figure(figsize=(8, 5))
    # sns.barplot(x=list(accuracies.keys()), y=list(accuracies.values()))
    # plt.ylabel('Accuracy')
    # plt.title('Model Accuracy Comparison (with Poll-Based Ensemble)')
    # plt.ylim(0, 1)
    # plt.xticks(rotation=45)
    # plt.tight_layout()
    # plt.show()

    # from sklearn.model_selection import cross_val_score
    # from sklearn.metrics import roc_curve, auc

    # # Prepare ROC plot
    # plt.figure(figsize=(8, 6))

    # for name, model in models.items():
    #     # Only for classifiers that support probability or decision scores
    #     if hasattr(model, "predict_proba"):
    #         if name in ['KNN', 'SVM']:
    #             probs = model.predict_proba(X_test)[:, 1]
    #         else:
    #             probs = model.predict_proba(X_test)[:, 1]
    #     elif hasattr(model, "decision_function"):
    #         if name == "SVM":
    #             probs = model.decision_function(X_test)
    #         else:
    #             continue  # skip if decision_function is not available
    #     else:
    #         continue  # skip models that can't produce scores

    #     # ROC curve
    #     fpr, tpr, _ = roc_curve(y_test, probs)
    #     roc_auc = auc(fpr, tpr)
    #     plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')

    # # Plot formatting
    # plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    # plt.title('ROC Curve Comparison')
    # plt.xlabel('False Positive Rate')
    # plt.ylabel('True Positive Rate')
    # plt.legend()
    # plt.grid(True)
    # plt.tight_layout()
    # plt.show()

    # # Cross-Validation
    # print("\nCross-Validation Results (5-Fold):")
    # for name, model in models.items():
    #     if name in ['KNN', 'SVM']:
    #         X_cv = X_train
    #     else:
    #         X_cv = X_train
    #     scores = cross_val_score(model, X_cv, y_train, cv=5, scoring='accuracy')
    #     print(f"{name}: Mean Accuracy = {scores.mean():.4f}, Std = {scores.std():.4f}")
    
    # return accuracies

In [ ]:
def run_models(df, run_PCA=True, PCA_n_components=100, test_size=0.2) -> dict:
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.naive_bayes import GaussianNB
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.decomposition import PCA
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns

    # 1. Prepare features and target
    X = df.drop(['mark', 'id', 'post_time'], axis=1)
    y = df['mark']

    # 2. Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    # 3. Standardize
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # 4. PCA (optional)
    if run_PCA:
        pca = PCA(n_components=PCA_n_components)
        X_train = pca.fit_transform(X_train)
        X_test = pca.transform(X_test)
        print(f"PCA reduced the feature count from {X.shape[1]} to {X_train.shape[1]}")

    # 5. Initialize models
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Naive Bayes': GaussianNB(),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'SVM': SVC(),
        'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
        'Random Forest': RandomForestClassifier(random_state=42, max_depth=10)
    }

    # 6. Evaluation results
    accuracies = {}

    for name, model in models.items():
        print(f"\n====== {name} ======")
        model.fit(X_train, y_train)

        # In-sample (training) predictions
        y_train_pred = model.predict(X_train)
        train_acc = accuracy_score(y_train, y_train_pred)
        print(f"[Train] Accuracy: {train_acc:.4f}")
        print("[Train] Classification Report:\n", classification_report(y_train, y_train_pred))

        # Out-of-sample (testing) predictions
        y_test_pred = model.predict(X_test)
        test_acc = accuracy_score(y_test, y_test_pred)
        accuracies[name] = test_acc
        print(f"[Test] Accuracy: {test_acc:.4f}")
        print("[Test] Classification Report:\n", classification_report(y_test, y_test_pred))

        # Plot confusion matrices
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.heatmap(confusion_matrix(y_train, y_train_pred), annot=True, fmt="d", ax=axes[0], cmap="Blues")
        axes[0].set_title(f"{name} - Train Confusion Matrix")
        axes[0].set_xlabel("Predicted")
        axes[0].set_ylabel("Actual")

        sns.heatmap(confusion_matrix(y_test, y_test_pred), annot=True, fmt="d", ax=axes[1], cmap="Oranges")
        axes[1].set_title(f"{name} - Test Confusion Matrix")
        axes[1].set_xlabel("Predicted")
        axes[1].set_ylabel("Actual")

        plt.tight_layout()
        plt.show()

    return accuracies

## Step 4: 跑模型的主程式

In [ ]:
from typing import Literal
def main(companyname, test_size=0.2, run_PCA=True, PCA_n_components=100, num_keywords=200, keyword_method: Literal["frequency", "tfidf", "lift"] = "tfidf", final_data_method: Literal["count", "contain"] = "count") -> dict:
    print(f"running models for {companyname}...")
    fileID_contain_companyname = get_fileID_contain_companyname(companyname)
    print("creating cleaned df...")
    cleaned_df = create_cleaned_df(fileID_contain_companyname, companyname)
    print("creating df_listed_selected...")
    df_listed_selected = create_df_listed_selected(df_listed, companyname)
    print("updating mark...")
    cleaned_df = update_mark(cleaned_df, df_listed_selected)
    print("creating keywords...")

    # num_keywords = int(num_keywords / 2)
    
    df_bull = cleaned_df[cleaned_df["mark"] == 1].copy()
    df_bear = cleaned_df[cleaned_df["mark"] == 0].copy()
    df_bull.reset_index(drop=True, inplace=True)
    df_bear.reset_index(drop=True, inplace=True)
    bear_key_words = []


    if keyword_method == "frequency":
        bull_key_words = get_keywords(df_bull, num_keywords, companyname)
        # bear_key_words = get_keywords(df_bear, num_keywords, companyname)
        key_words = list(set(bull_key_words + bear_key_words))
    elif keyword_method == "tfidf":
        bull_key_words = get_keywords_tfidf(df_bull, num_keywords, companyname)
        # bear_key_words = get_keywords_tfidf(df_bear, num_keywords, companyname)
        key_words = list(set(bull_key_words + bear_key_words))
    elif keyword_method == "lift":
        bull_key_words = get_keywords_lift(df_bull, num_keywords, companyname)
        # bear_key_words = get_keywords_lift(df_bear, num_keywords, companyname)
        key_words = list(set(bull_key_words + bear_key_words))
    
    print("creating final data...")
    df = create_final_data(cleaned_df, key_words, companyname, keyword_method, method=final_data_method)
    
    print("running models...")
    result = run_models(df, run_PCA=run_PCA, PCA_n_components=PCA_n_components, test_size=test_size)
    return result

In [ ]:
nums = [200]
results = []

for num in nums:
    print(f"num_keywords: {num}")
    result = main(
        companyname="龍德造船",
        test_size=0.2,
        run_PCA=True, 
        PCA_n_components=10, 
        num_keywords=num, 
        keyword_method="frequency",
        final_data_method="contain"
    )
    results.append(result)
    print("=========================================")

df = pd.DataFrame(results)
df